Data Preprocessing

In [ ]:
import findspark
findspark.init()

from pyspark.sql import SparkSession
from pyspark.sql.types import FloatType, IntegerType
import pyspark.sql.functions as F

# Define Spark Obj and allocate memory
spark = SparkSession.builder \
    .appName("AmazonBooks_ETL") \
    .master("local[*]") \
    .config("spark.driver.memory", "8g") \
    .getOrCreate()

print("Spark Engine is ONLINE. Ready to process Big Data.")

# Now 'spark' exists, so it works.
df_ratings_raw = spark.read.csv("dataset/books_rating.csv", header=True, inferSchema=True)
df_books_raw = spark.read.csv("dataset/books_data.csv", header=True, inferSchema=True)

print("Data loaded successfully into memory.")

# 2.1 Drop heavy text fields and unused links to save memory
df_books_clean = df_books_raw.drop("image", "previewLink", "infoLink", "description", "ratingsCount")

# 2.2 Drop rows with null Titles (essential for the upcoming Join)
df_books_clean = df_books_clean.dropna(subset=["Title"])

# 2.3 Convert stringified arrays (e.g., "['Fiction']") to PySpark ArrayType
# Use regex to remove brackets and quotes, then split by comma
df_books_clean = df_books_clean \
    .withColumn("categories_array", F.split(F.regexp_replace("categories", r"\['|'\]|\"|\[|\]", ""), ", ")) \
    .withColumn("authors_array", F.split(F.regexp_replace("authors", r"\['|'\]|\"|\[|\]", ""), ", ")) \
    .drop("categories", "authors")


# 3.1 Rename columns with slashes to prevent SQL syntax errors downstream
df_ratings_clean = df_ratings_raw \
    .withColumnRenamed("review/score", "rating") \
    .withColumnRenamed("review/helpfulness", "helpfulness") \
    .withColumnRenamed("review/time", "review_time")

# 3.2 Drop heavy NLP columns and irrelevant user identifiers
df_ratings_clean = df_ratings_clean.drop("review/text", "review/summary", "profileName", "Id", "User_id")

# 3.3 Clean 'Price' (force cast to Float) and handle nulls
df_ratings_clean = df_ratings_clean \
    .withColumn("Price", F.col("Price").cast(FloatType())) \
    .dropna(subset=["Price", "rating", "Title"])

# 3.4 Convert Unix timestamp to DateType (YYYY-MM-DD format)
df_ratings_clean = df_ratings_clean \
    .withColumn("review_date", F.to_date(F.from_unixtime("review_time"))) \
    .drop("review_time")

# 3.5 Split 'helpfulness' (e.g., "7/9") into two distinct integer columns
helpfulness_split = F.split("helpfulness", "/")
df_ratings_clean = df_ratings_clean \
    .withColumn("helpful_votes", helpfulness_split.getItem(0).cast(IntegerType())) \
    .withColumn("total_votes", helpfulness_split.getItem(1).cast(IntegerType())) \
    .drop("helpfulness")


# 4 Print schemas to verify the transformations before executing the Join
print("--- Cleaned Books Schema ---")
df_books_clean.printSchema()

print("--- Cleaned Ratings Schema ---")
df_ratings_clean.printSchema()

import pyspark.sql.functions as F
from pyspark.sql.types import FloatType

# 5
# Fix the 'rating' column type (String -> Float) before joining
df_ratings_clean = df_ratings_clean.withColumn("rating", F.col("rating").cast(FloatType()))

# Perform the Broadcast Join to combine ratings with book metadata
df_joined = df_ratings_clean.join(F.broadcast(df_books_clean), on="Title", how="inner")

# Persist the joined table in memory for fast EDA and subsequent queries
df_joined.cache()



# 6: Exploratory Data Analysis (EDA)
print("\n" + "="*40)
print("EXPLORATORY DATA ANALYSIS (EDA)")
print("="*40)

# 6.1 The "Dimensions" (Row and Column counts)
# Note: count() triggers the actual execution of the lazy join
total_rows = df_joined.count()
total_cols = len(df_joined.columns)
print(f"Dataset Shape: {total_rows:,} rows | {total_cols} columns")

# 6.2 Numerical Summary (Mean, Stddev, Min, Max)
# Calculate core statistics for Price, rating, and helpfulness without printing raw data
print("\n--- Numerical Summary ---")
df_joined.select("Price", "rating", "helpful_votes", "total_votes") \
         .summary("mean", "stddev", "min", "max") \
         .show()

# 6.3 Temporal Range
# Find the earliest and latest reviews in the dataset
print("--- Temporal Range ---")
df_joined.select(
    F.min("review_date").alias("Earliest_Review"),
    F.max("review_date").alias("Latest_Review")
).show()

# 6.4 Missing Values Check
# Count how many nulls remain in the key columns
print("--- Missing Values Check ---")
df_joined.select([F.count(F.when(F.isnull(c), c)).alias(c) for c in ["Price", "rating", "categories_array"]]).show()

# 6.5 Peek at the Data (Top 3 rows only)
print("--- Data Preview (Top 3 rows) ---")
df_joined.select("Title", "Price", "rating", "categories_array").show(3, truncate=False)

In [ ]:
# 7 Post-EDA Data Correction

# 1. Drop the newly unmasked null ratings
df_final = df_joined.dropna(subset=["rating"])

# 2. Filter out impossible ratings (valid range: 1.0 to 5.0)
df_final = df_final.filter((F.col("rating") >= 1.0) & (F.col("rating") <= 5.0))

# 3. Filter out "1970-01-01" default timestamps
df_final = df_final.filter(F.col("review_date") > "1970-01-01")

# 4. Memory Management: Cache the new clean DataFrame and release the old one
df_final.cache()
df_joined.unpersist() 

print(f"Bugs squashed! Final clean dataset size: {df_final.count():,} rows.")

In [ ]:
# 8 Post-Correction Validation (Quick EDA)

print("\n" + "="*45)
print("POST-CLEANING VALIDATION CHECK")
print("="*45)

# 1. Verify Dataset Size
print(f"Final Dataset Size: {df_final.count():,} rows")

# 2. Verify 'rating' bounds (Should strictly be between 1.0 and 5.0)
print("\n--- Rating Bounds Check ---")
df_final.select("rating").summary("min", "max").show()

# 3. Verify 'review_date' bounds (Should not include 1970-01-01)
print("--- Date Bounds Check ---")
df_final.select(
    F.min("review_date").alias("Earliest_Valid_Date"),
    F.max("review_date").alias("Latest_Date")
).show()

# 4. Verify Missing Values in core columns (Should be exactly 0)
print("--- Null Check ---")
df_final.select([
    F.count(F.when(F.isnull(c), c)).alias(c) 
    for c in ["Price", "rating"]
]).show()

In [ ]:
import pyspark.sql.functions as F

# Define a list of format-related tags that should be excluded from genre analysis
format_tags_to_exclude = [
    "Audiobooks", 
    "Audible Audiobooks", 
    "Books on CD", 
    "Kindle Store", 
    "Audio CD",
    "Books" 
]

# Explode the categories array so books with multiple categories are counted accurately
df_exploded = df_final.withColumn("single_category", F.explode("categories_array"))

# Filter out the physical or digital format tags to isolate pure content genres
df_genres_only = df_exploded.filter(~F.col("single_category").isin(format_tags_to_exclude))

# Group by the true genre and calculate average rating and total volume
category_metrics = df_genres_only.groupBy("single_category") \
    .agg(
        F.round(F.avg("rating"), 2).alias("avg_rating"),
        F.count("*").alias("total_reviews")
    )

# Filter out niche categories with too few reviews to ensure statistical significance
# Sort by highest average rating first, using total reviews as a tie-breaker
top_categories = category_metrics.filter("total_reviews > 1000") \
    .orderBy(F.desc("avg_rating"), F.desc("total_reviews"))

# Display the top 10 true genres in the console
print("Top Categories by Average Rating (Formats Excluded):")
top_categories.show(10, truncate=False)

# Export the aggregated data to CSV for reporting
output_file_q1 = "Result/Q1_Category_Preference.csv"
top_categories.toPandas().to_csv(output_file_q1, index=False)
print(f"Results exported to: {output_file_q1}")

In [ ]:
# Explode the authors array to evaluate each author's individual brand power
df_authors = df_final.withColumn("author", F.explode("authors_array"))

# Calculate total reviews and average rating for each specific author
author_stats = df_authors.groupBy("author") \
    .agg(
        F.count("*").alias("total_reviews"),
        F.avg("rating").alias("author_avg_rating")
    )

# Segment authors into popularity tiers to answer if best-sellers maintain quality
author_tiers = author_stats.withColumn(
    "popularity_tier",
    F.when(F.col("total_reviews") < 100, "1. Niche (< 100 reviews)")
     .when((F.col("total_reviews") >= 100) & (F.col("total_reviews") < 1000), "2. Established (100 - 999 reviews)")
     .otherwise("3. Best-Selling (1000+ reviews)")
)

# Aggregate the final metrics by these popularity tiers
tier_metrics = author_tiers.groupBy("popularity_tier") \
    .agg(
        F.round(F.avg("author_avg_rating"), 2).alias("avg_tier_rating"),
        F.count("author").alias("total_authors_in_tier")
    ).orderBy("popularity_tier")

# Display the findings in the console
print("Relationship Between Author Popularity and Rating Quality:")
tier_metrics.show(truncate=False)

# Export the aggregated data to CSV for reporting
output_file_q2 = "Result/Q2_Brand_Popularity_Quality.csv"
tier_metrics.toPandas().to_csv(output_file_q2, index=False)
print(f"Results exported to: {output_file_q2}")

In [ ]:
# Group by Title to calculate the absolute review volume for each specific book.
# We use F.first() to retain the author information for display without duplicating rows.
market_leaders = df_final.groupBy("Title") \
    .agg(
        F.count("*").alias("total_reviews"),
        F.first("authors_array").alias("authors"),
        F.round(F.avg("rating"), 2).alias("avg_rating")
    )

# Sort strictly by total reviews in descending order to find the most dominant titles
top_market_leaders = market_leaders.filter(F.col("Title").isNotNull()) \
    .orderBy(F.desc("total_reviews"))

# Display the top 10 market leaders in the console
print("Market Leaders - Top Titles by Review Volume:")
top_market_leaders.select("Title", "authors", "total_reviews", "avg_rating").show(10, truncate=False)

# Export the aggregated data to CSV for reporting
output_file_q3 = "Result/Q3_Market_Leaders.csv"
top_market_leaders.toPandas().to_csv(output_file_q3, index=False)
print(f"Results exported to: {output_file_q3}")

In [ ]:
# Reusing the format exclusion list to ensure consistency across the project
format_tags_to_exclude = [
    "Audiobooks", 
    "Audible Audiobooks", 
    "Books on CD", 
    "Kindle Store", 
    "Audio CD",
    "Books" 
]

# Explode the categories array 
df_exploded_price = df_final.withColumn("genre", F.explode("categories_array"))

# Filter out the format tags to isolate pure content genres
df_genres_only = df_exploded_price.filter(~F.col("genre").isin(format_tags_to_exclude))

# Filter out rows where the price is null
df_clean_price = df_genres_only.filter(F.col("Price").isNotNull())

# Group by the true genre and calculate the average price and total volume
genre_pricing = df_clean_price.groupBy("genre") \
    .agg(
        F.round(F.avg("Price"), 2).alias("avg_price"),
        F.count("*").alias("total_books")
    )

# Filter out extremely niche genres and sort by the most expensive
top_genre_pricing = genre_pricing.filter("total_books > 1000") \
    .orderBy(F.desc("avg_price"))

# Display the clean results in the console
print("Pricing Strategy - Average Price by Genre (Formats Excluded):")
top_genre_pricing.show(10, truncate=False)

# Export the aggregated data to CSV
output_file_q4 = "Result/Q4_Pricing_Strategy.csv"
top_genre_pricing.toPandas().to_csv(output_file_q4, index=False)
print(f"Results exported to: {output_file_q4}")

In [ ]:
# Filter out rows where the price is missing to ensure accurate calculations
df_valid_prices = df_final.filter(F.col("Price").isNotNull())

# Create logical price buckets to make the trend easier to analyze and visualize
df_price_tiers = df_valid_prices.withColumn(
    "price_tier",
    F.when(F.col("Price") < 10.0, "1. Under $10")
     .when((F.col("Price") >= 10.0) & (F.col("Price") < 20.0), "2. $10 - $19.99")
     .when((F.col("Price") >= 20.0) & (F.col("Price") < 50.0), "3. $20 - $49.99")
     .when((F.col("Price") >= 50.0) & (F.col("Price") < 100.0), "4. $50 - $99.99")
     .otherwise("5. $100 and above")
)

# Group by the newly created price tier and calculate average rating and review volume
price_metrics = df_price_tiers.groupBy("price_tier") \
    .agg(
        F.round(F.avg("rating"), 2).alias("avg_rating"),
        F.count("*").alias("total_reviews")
    ).orderBy("price_tier")

# Display the findings in the console
print("Price vs. Satisfaction - Average Rating by Price Tier:")
price_metrics.show(truncate=False)

# Export the aggregated data to CSV for reporting
output_file_q5 = "Result/Q5_Price_vs_Satisfaction.csv"
price_metrics.toPandas().to_csv(output_file_q5, index=False)
print(f"Results exported to: {output_file_q5}")

In [ ]:
# Filter out records where publisher information is missing to ensure data quality
df_publishers = df_final.filter(F.col("publisher").isNotNull())

# Group by publisher to calculate market share metrics
# countDistinct evaluates the breadth of their portfolio (total books published)
# count evaluates the overall reader engagement (total review volume)
publisher_metrics = df_publishers.groupBy("publisher") \
    .agg(
        F.countDistinct("Title").alias("total_books_published"),
        F.count("*").alias("total_reviews"),
        F.round(F.avg("rating"), 2).alias("avg_rating")
    )

# Filter out minor publishers to focus on major market players
# Sort by reader engagement (total reviews) to define the largest market share
top_publishers = publisher_metrics.filter("total_reviews > 2000") \
    .orderBy(F.desc("total_reviews"))

# Display the top 10 publishers in the console
print("Publisher Power - Market Share by Engagement and Portfolio Size:")
top_publishers.show(10, truncate=False)

# Export the aggregated data to CSV for reporting
output_file_q6 = "Result/Q6_Publisher_Power.csv"
top_publishers.toPandas().to_csv(output_file_q6, index=False)
print(f"Results exported to: {output_file_q6}")

In [ ]:
# Extract the year from the review_date column
df_with_year = df_final.withColumn("review_year", F.year("review_date"))

# Filter for the specific time range requested (1996 to 2014)
# Ensure we only include rows with valid years
df_filtered_years = df_with_year.filter(
    (F.col("review_year").isNotNull()) & 
    (F.col("review_year") >= 1996) & 
    (F.col("review_year") <= 2014)
)

# Group by year to calculate the average sentiment (rating) and review volume
yearly_trends = df_filtered_years.groupBy("review_year") \
    .agg(
        F.round(F.avg("rating"), 2).alias("avg_rating"),
        F.count("*").alias("total_reviews")
    )

# Sort the results chronologically
chronological_trends = yearly_trends.orderBy("review_year")

# Display the results in the console
print("Temporal Trends - Sentiment Evolution (1996-2014):")
chronological_trends.show(20, truncate=False)

# Export the aggregated data to CSV for reporting
output_file_q7 = "Result/Q7_Temporal_Trends.csv"
chronological_trends.toPandas().to_csv(output_file_q7, index=False)
print(f"Results exported to: {output_file_q7}")

In [ ]:
# Extract the month from the review date to analyze seasonal patterns
df_season = df_final.withColumn("review_month", F.month("review_date"))

# Filter out rows with invalid or missing dates
df_valid_months = df_season.filter(F.col("review_month").isNotNull())

# Group by month and calculate the total review volume and average rating
monthly_metrics = df_valid_months.groupBy("review_month") \
    .agg(
        F.count("*").alias("total_reviews"),
        F.round(F.avg("rating"), 2).alias("avg_rating")
    )

# Sort chronologically from January (1) to December (12)
seasonal_trends = monthly_metrics.orderBy("review_month")

# Display the 12-month cycle in the console
print("Seasonality - Consumer Engagement by Month:")
seasonal_trends.show(12, truncate=False)

# Export the aggregated data to CSV for reporting
output_file_q9 = "Result/Q9_Seasonality.csv"
seasonal_trends.toPandas().to_csv(output_file_q9, index=False)
print(f"Results exported to: {output_file_q9}")

In [ ]:
# Group by Title to aggregate metrics for each specific book
book_hype_metrics = df_final.groupBy("Title") \
    .agg(
        F.count("*").alias("total_reviews"),
        F.round(F.avg("rating"), 2).alias("avg_rating")
    )

# Filter for books that generated massive hype but failed to deliver quality
# Here we define "hype" as > 500 reviews and "low satisfaction" as < 3.5 average rating
hype_factor_books = book_hype_metrics.filter(
    (F.col("total_reviews") > 500) & 
    (F.col("avg_rating") < 4)
).orderBy(F.desc("total_reviews"), F.asc("avg_rating"))

# Display the top 10 titles fitting the hype factor criteria in the console
print("The Hype Factor - High Popularity, Low Satisfaction:")
hype_factor_books.show(10, truncate=False)

# Export the aggregated data to CSV for reporting
output_file_q10 = "Result/Q10_The_Hype_Factor.csv"
hype_factor_books.toPandas().to_csv(output_file_q10, index=False)
print(f"Results exported to: {output_file_q10}")

In [ ]:
from pyspark.sql.types import IntegerType

# Define the path to dataset
raw_data_path = "dataset/Books_rating.csv" 

# 1. Column Pruning & Renaming: 
# Select using the exact column names with slashes, and rename them instantly to avoid syntax issues later.
df_raw_q8 = spark.read.csv(raw_data_path, header=True, inferSchema=True) \
    .select(
        F.col("review/text").alias("review_text"), 
        F.col("review/helpfulness")
    )

# 2. Parsing the Helpfulness String:
# The format is typically "X/Y" (e.g., "3/5"). We split by "/" and cast to integers.
df_parsed = df_raw_q8.withColumn(
    "helpful_votes", 
    F.split(F.col("review/helpfulness"), "/").getItem(0).cast(IntegerType())
).withColumn(
    "total_votes", 
    F.split(F.col("review/helpfulness"), "/").getItem(1).cast(IntegerType())
)

# 3. Filter Noise and Nulls:
# Drop empty texts and enforce a strict threshold (>= 10 total votes) to eliminate noise.
df_q8_clean = df_parsed.filter(
    F.col("review_text").isNotNull() & 
    F.col("total_votes").isNotNull() & 
    (F.col("total_votes") >= 10)
)

# 4. Feature Engineering: Word Count and Helpfulness Ratio
# Count words by splitting spaces, and calculate the ratio.
df_q8_metrics = df_q8_clean.withColumn(
    "review_length_words", F.size(F.split(F.col("review_text"), "\\s+"))
).withColumn(
    "helpfulness_ratio", F.col("helpful_votes") / F.col("total_votes")
)

# 5. Mathematical Correlation (The ultimate answer)
correlation_q8 = df_q8_metrics.corr("review_length_words", "helpfulness_ratio")

print("Review Quality - Correlation Analysis:")
print(f"Pearson Correlation (Word Count vs. Helpfulness): {correlation_q8:.4f}")

# 6. Binning for Visualization: 
# Group review lengths into readable buckets.
df_length_tiers = df_q8_metrics.withColumn(
    "length_tier",
    F.when(F.col("review_length_words") < 50, "1. Short (< 50 words)")
     .when((F.col("review_length_words") >= 50) & (F.col("review_length_words") < 100), "2. Medium (50 - 99 words)")
     .when((F.col("review_length_words") >= 100) & (F.col("review_length_words") < 300), "3. Long (100 - 299 words)")
     .when((F.col("review_length_words") >= 300) & (F.col("review_length_words") < 500), "4. Essay (300 - 499 words)")
     .otherwise("5. Novel (500+ words)")
)

length_analysis = df_length_tiers.groupBy("length_tier") \
    .agg(
        F.round(F.avg("helpfulness_ratio"), 4).alias("avg_helpfulness_ratio"),
        F.count("*").alias("total_reviews_in_tier")
    ).orderBy("length_tier")

print("\nReview Quality - Helpfulness by Word Count Tier:")
length_analysis.show(truncate=False)

# Export the aggregated data to CSV for reporting
output_file_q8 = "Result/Q8_Review_Quality_Length.csv"
length_analysis.toPandas().to_csv(output_file_q8, index=False)
print(f"Results exported to: {output_file_q8}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set global plotting style
sns.set_theme(style="whitegrid", context="talk") 

# Define a professional color palette inspired by corporate reporting
# Using dark blue and orange for high contrast
AMZ_PALETTE = ["#232F3E", "#FF9900", "#146EB4", "#E47911", "#F2F2F2"]
sns.set_palette(sns.color_palette(AMZ_PALETTE))

# Improve chart resolution for high-quality export
%config InlineBackend.figure_format = 'retina'

# Ensure fonts are rendered correctly without specialized symbols
plt.rcParams['font.sans-serif'] = ['Arial']
plt.rcParams['axes.unicode_minus'] = False 

print("Visualization environment initialized. Global styles applied.")

In [ ]:
import pandas as pd
#import matplotlib.subplots
import matplotlib.pyplot as plt
import seaborn as sns

q1_data = pd.read_csv("Result/Q1_Category_Preference.csv", encoding='utf-8-sig')

q1_data.columns = q1_data.columns.str.strip().str.lower()

if 'unnamed: 0' in q1_data.columns:
    q1_data = q1_data.drop(columns=['unnamed: 0'])

rename_mapping = {}
if 'category' in q1_data.columns and 'single_category' not in q1_data.columns:
    rename_mapping['category'] = 'single_category'
if 'average_rating' in q1_data.columns and 'avg_rating' not in q1_data.columns:
    rename_mapping['average_rating'] = 'avg_rating'
if 'rating' in q1_data.columns and 'avg_rating' not in q1_data.columns:
    rename_mapping['rating'] = 'avg_rating'

if rename_mapping:
    q1_data.rename(columns=rename_mapping, inplace=True)

# 3. Data Transformation
# Sort descending by rating and focus on the top 15 categories for visual clarity
q1_plot_data = q1_data.sort_values(by="avg_rating", ascending=False).head(15).reset_index(drop=True)

# 4. Visualization Setup
plt.figure(figsize=(12, 8))
sns.set_context("talk", font_scale=0.9)

# Create a horizontal bar chart
# Color intensity (hue/palette) represents visual depth
barplot = sns.barplot(
    data=q1_plot_data, 
    x="avg_rating", 
    y="single_category", 
    palette="viridis", 
    edgecolor="black"
)

# 5. Dynamic Annotations
# Calculate the minimum and maximum ratings to establish dynamic offsets and axis limits
min_rating = q1_plot_data["avg_rating"].min()
max_rating = q1_plot_data["avg_rating"].max()
offset = (max_rating - min_rating) * 0.05 if max_rating > min_rating else 0.02

# Add numerical labels to the end of each bar for precision
for p in barplot.patches:
    width = p.get_width()
    plt.text(
        width + (offset * 0.5), 
        p.get_y() + p.get_height() / 2, 
        f'{width:.2f}', 
        ha="left", 
        va="center",
        fontsize=12,
        weight='bold',
        color="#334155"
    )

# 6. Professional Titles and Labels
plt.title("Top 15 Categories by Average Rating: Market Quality Index", pad=20, weight='bold', fontsize=18)
plt.xlabel("Average Customer Rating (Scale: 1-5)", fontsize=14, weight='bold', labelpad=10)
plt.ylabel("Book Categories", fontsize=14, weight='bold')

# Narrow the X-axis range dynamically to emphasize differences between high-performing categories
# This replaces the hardcoded (4.3, 4.55) to prevent breaking if the data changes slightly
plt.xlim(max(0, min_rating - (offset * 2)), max_rating + (offset * 3)) 

# 7. Layout Optimization
# Remove top and right spines for a clean "modern" look
sns.despine()
plt.tight_layout()

plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


q2_data = pd.read_csv("Result/Q2_Brand_Popularity_Quality.csv", encoding='utf-8-sig')


q2_data.columns = q2_data.columns.str.strip().str.lower()

# Safely drop the pandas default index column if it was exported to the CSV
if 'unnamed: 0' in q2_data.columns:
    q2_data = q2_data.drop(columns=['unnamed: 0'])

rename_mapping = {}
if 'tier' in q2_data.columns and 'popularity_tier' not in q2_data.columns:
    rename_mapping['tier'] = 'popularity_tier'
if 'author_tier' in q2_data.columns and 'popularity_tier' not in q2_data.columns:
    rename_mapping['author_tier'] = 'popularity_tier'
if 'avg_rating' in q2_data.columns and 'avg_tier_rating' not in q2_data.columns:
    rename_mapping['avg_rating'] = 'avg_tier_rating'
if 'rating' in q2_data.columns and 'avg_tier_rating' not in q2_data.columns:
    rename_mapping['rating'] = 'avg_tier_rating'
if 'average_rating' in q2_data.columns and 'avg_tier_rating' not in q2_data.columns:
    rename_mapping['average_rating'] = 'avg_tier_rating'

if rename_mapping:
    q2_data.rename(columns=rename_mapping, inplace=True)

# 3. Visualization Setup
# Set a wider figure to accommodate text and give a professional presentation feel
plt.figure(figsize=(12, 6))
sns.set_context("talk", font_scale=0.9)

# Create a bar plot with improved spacing
# We set 'width' to ensure bars don't look crowded
barplot_q2 = sns.barplot(
    data=q2_data, 
    x="popularity_tier", 
    y="avg_tier_rating", 
    palette="viridis",
    edgecolor="black",
    width=0.6
)

# 4. Dynamic Annotations
# Add data labels with enough clearance
for i, val in enumerate(q2_data["avg_tier_rating"]):
    plt.text(
        i, 
        val + 0.1, # Added a slight vertical offset to avoid touching the bar edge
        f'{val:.2f}', 
        ha="center", 
        va="bottom",
        fontsize=13,
        weight='bold',
        color="#334155"
    )

# 5. Professional Titles and Labels
plt.title("Author Popularity vs. Average Quality Rating (Refined Scale)", pad=20, weight='bold', fontsize=18)
plt.xlabel("Author Popularity Tier", fontsize=14, weight='bold', labelpad=10)
plt.ylabel("Average Rating", fontsize=14, weight='bold')

# Use a more natural Y-axis range to avoid visual distortion 
# Setting max to 5.5 instead of 5.0 gives the text annotations room to breathe
plt.ylim(0, 5.5) 

# Rotate X-axis labels to prevent overlap if necessary
plt.xticks(rotation=0)

# 6. Layout Optimization
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Bulletproof Data Loading
# Use utf-8-sig to handle potential Windows BOM characters
# Adjust the path if the file is not located inside the "Result/" directory
q3_data = pd.read_csv("Result/Q3_Market_Leaders.csv", encoding='utf-8-sig')

# 2. Data Cleaning & Smart Column Mapping
# Strip hidden spaces and convert all columns to lowercase to prevent KeyErrors
q3_data.columns = q3_data.columns.str.strip().str.lower()

# Safely drop the pandas default index column if it was exported to the CSV
if 'unnamed: 0' in q3_data.columns:
    q3_data = q3_data.drop(columns=['unnamed: 0'])

# Smart mapping to ensure we have the expected columns ('title', 'total_reviews', 'avg_rating')
rename_mapping = {}
if 'book_title' in q3_data.columns and 'title' not in q3_data.columns:
    rename_mapping['book_title'] = 'title'
if 'review_count' in q3_data.columns and 'total_reviews' not in q3_data.columns:
    rename_mapping['review_count'] = 'total_reviews'
if 'average_rating' in q3_data.columns and 'avg_rating' not in q3_data.columns:
    rename_mapping['average_rating'] = 'avg_rating'
if 'rating' in q3_data.columns and 'avg_rating' not in q3_data.columns:
    rename_mapping['rating'] = 'avg_rating'

if rename_mapping:
    q3_data.rename(columns=rename_mapping, inplace=True)

# 3. Data Transformation
# Sort descending by volume and focus on the top 15 market leaders
q3_plot_data = q3_data.sort_values(by="total_reviews", ascending=False).head(15).reset_index(drop=True)

# Truncate titles longer than 50 characters to keep the layout clean
# Convert to string first to handle potential nulls or numeric titles
q3_plot_data['title_short'] = q3_plot_data['title'].apply(
    lambda x: str(x)[:47] + '...' if pd.notna(x) and len(str(x)) > 50 else str(x)
)

# 4. Visualization Setup
plt.figure(figsize=(14, 10))
sns.set_context("talk", font_scale=0.9)

# Define limits for the color mapping based on the actual ratings
vmin_val = q3_plot_data['avg_rating'].min()
vmax_val = q3_plot_data['avg_rating'].max()

# Create the horizontal bar chart
# We use horizontal bar length to represent volume and color (hue) to represent satisfaction
barplot_q3 = sns.barplot(
    data=q3_plot_data, 
    x="total_reviews", 
    y="title_short", 
    hue="avg_rating",
    palette="RdYlGn", 
    hue_norm=(vmin_val, vmax_val),
    dodge=False,
    edgecolor="black",
    legend=False
)

# Add a custom colorbar to explain the rating heatmap colors
norm = plt.Normalize(vmin_val, vmax_val)
sm = plt.cm.ScalarMappable(cmap="RdYlGn", norm=norm)
cbar = plt.gcf().colorbar(sm, ax=plt.gca(), pad=0.02)
cbar.set_label("Average Customer Rating (Satisfaction)", weight='bold')

# 5. Dynamic Annotations
# Calculate a dynamic offset based on the maximum volume to prevent overlap
max_reviews = q3_plot_data['total_reviews'].max()
offset = max_reviews * 0.015

for i, val in enumerate(q3_plot_data["total_reviews"]):
    plt.text(
        val + offset, 
        i, 
        f'{int(val)}', 
        va="center", 
        fontsize=12,
        weight='semibold',
        color="#334155"
    )

# 6. Professional Formatting
plt.title("Market Leaders: Review Volume and Satisfaction Analysis", pad=20, weight='bold', fontsize=18)
plt.xlabel("Total Review Volume (Market Presence)", fontsize=14, weight='bold')
plt.ylabel("Top 15 Book Titles (Truncated for Clarity)", fontsize=14, weight='bold')

# Expand X-axis limit slightly to accommodate the text labels
plt.xlim(0, max_reviews * 1.15)

# Layout Optimization
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Bulletproof Data Loading
# Use utf-8-sig to eliminate hidden Windows BOM characters
# Adjust the path to "Q4_Pricing_Strategy.csv" if it is not inside the "Result/" folder
q4_data = pd.read_csv("Result/Q4_Pricing_Strategy.csv", encoding='utf-8-sig')

# 2. Data Cleaning & Smart Column Mapping
# Strip hidden spaces and convert all columns to lowercase to prevent KeyErrors
q4_data.columns = q4_data.columns.str.strip().str.lower()

# Safely drop the pandas default index column if it was exported to the CSV
if 'unnamed: 0' in q4_data.columns:
    q4_data = q4_data.drop(columns=['unnamed: 0'])

# Smart mapping to ensure we have the expected columns ('genre', 'avg_price')
rename_mapping = {}
if 'category' in q4_data.columns and 'genre' not in q4_data.columns:
    rename_mapping['category'] = 'genre'
if 'average_price' in q4_data.columns and 'avg_price' not in q4_data.columns:
    rename_mapping['average_price'] = 'avg_price'
if 'price' in q4_data.columns and 'avg_price' not in q4_data.columns:
    rename_mapping['price'] = 'avg_price'

if rename_mapping:
    q4_data.rename(columns=rename_mapping, inplace=True)

# Focus on the top 20 most expensive genres for a sharp business focus
# Ensure data is sorted descending by price before taking the top 20
q4_plot_data = q4_data.sort_values(by='avg_price', ascending=False).head(20).reset_index(drop=True)

# 3. Visualization Setup
# Initialize the plot with a professional size
plt.figure(figsize=(12, 10))
sns.set_context("talk", font_scale=0.9)

# Create a horizontal bar chart
# X-axis: Average Price, Y-axis: Genre
barplot_q4 = sns.barplot(
    data=q4_plot_data, 
    x="avg_price", 
    y="genre", 
    palette="flare",
    edgecolor="black"
)

# 4. Dynamic Annotations
# Add precise price labels to the end of each bar
# Calculate a dynamic offset based on the maximum price to prevent overlap
max_price = q4_plot_data['avg_price'].max()
offset = max_price * 0.02

for p in barplot_q4.patches:
    width = p.get_width()
    plt.text(
        width + offset, 
        p.get_y() + p.get_height() / 2, 
        f'${width:.2f}', 
        ha="left", 
        va="center",
        fontsize=12,
        weight='semibold',
        color="#334155"
    )

# 5. Professional Titles and Labels
plt.title("Pricing Strategy: Top 20 Genres by Average Book Price", pad=20, weight='bold', fontsize=18)
plt.xlabel("Average Price (USD)", fontsize=14, weight='bold')
plt.ylabel("Book Genre", fontsize=14, weight='bold')

# Extend X-axis limit slightly to accommodate the text labels
plt.xlim(0, max_price * 1.15)

# 6. Layout Optimization
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Step 1: Calculate the distribution of each star rating (1-5) per price tier
# We use the existing df_price_tiers from your ETL
q5_distribution = df_price_tiers.groupBy("price_tier", "rating") \
    .count() \
    .toPandas()

# Step 2: Pivot the data to get ratings as columns for stacking
# This creates a table where rows = price tiers and columns = star ratings
q5_pivot = q5_distribution.pivot(index='price_tier', columns='rating', values='count').fillna(0)

# Step 3: Normalize the data (convert raw counts to percentages)
# This ensures each bar totals 100%, making comparisons fair
q5_perc = q5_pivot.div(q5_pivot.sum(axis=1), axis=0) * 100

# Step 4: Plotting
plt.figure(figsize=(14, 8))
sns.set_context("talk", font_scale=0.9)

# Define a color palette from Red (Low Rating) to Green (High Rating)
colors = ["#d73027", "#f46d43", "#fdae61", "#a6d96a", "#1a9850"]

# Create the stacked bar chart
ax = q5_perc.plot(
    kind='barh', 
    stacked=True, 
    figsize=(14, 8), 
    color=colors, 
    edgecolor="white"
)

# Professional titles and labels
plt.title("Rating Distribution by Price Tier (Percentage Breakdown)", pad=20, weight='bold', fontsize=16)
plt.xlabel("Percentage of Total Reviews (%)", weight='bold')
plt.ylabel("Price Tier (USD)", weight='bold')

# Place the legend outside to avoid cluttering the chart
plt.legend(title="Star Rating", bbox_to_anchor=(1.05, 1), loc='upper left', frameon=True, edgecolor="black")

# Add percentage labels inside the bars for high-impact reporting
# We only label segments large enough to fit text (> 5%)
for p in ax.patches:
    width = p.get_width()
    if width > 5:
        x = p.get_x() + width / 2
        y = p.get_y() + p.get_height() / 2
        ax.annotate(
            f'{width:.1f}%', 
            (x, y), 
            ha='center', 
            va='center', 
            fontsize=12, 
            color='white', 
            weight='bold'
        )

# Layout Optimization
plt.subplots_adjust(right=0.85)
ax.grid(False)
sns.despine(right=False, top=True)

plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


q6_plot_data = pd.read_csv("Result/Q6_Publisher_Power.csv", encoding='utf-8-sig')

q6_plot_data.columns = q6_plot_data.columns.str.strip().str.lower()

if 'unnamed: 0' in q6_plot_data.columns:
    q6_plot_data = q6_plot_data.drop(columns=['unnamed: 0'])


rename_mapping = {}
if 'publisher_name' in q6_plot_data.columns and 'publisher' not in q6_plot_data.columns:
    rename_mapping['publisher_name'] = 'publisher'
if 'review_count' in q6_plot_data.columns and 'total_reviews' not in q6_plot_data.columns:
    rename_mapping['review_count'] = 'total_reviews'
if 'books_published' in q6_plot_data.columns and 'total_books_published' not in q6_plot_data.columns:
    rename_mapping['books_published'] = 'total_books_published'
if 'titles' in q6_plot_data.columns and 'total_books_published' not in q6_plot_data.columns:
    rename_mapping['titles'] = 'total_books_published'

if rename_mapping:
    q6_plot_data.rename(columns=rename_mapping, inplace=True)


q6_plot_data = q6_plot_data.sort_values(by="total_reviews", ascending=False).head(15).reset_index(drop=True)

# 3. Initialize the plot
plt.figure(figsize=(14, 10))
sns.set_context("talk", font_scale=0.9)

# OPTIMIZATION: Manually set vmin and vmax to "zoom in" on the rating differences.
vmin_val = 3.8
vmax_val = 4.5

# Use 'RdYlGn' palette for better intuitive sense (Red = lower, Green = higher)
barplot_q6 = sns.barplot(
    data=q6_plot_data, 
    x="total_reviews", 
    y="publisher", 
    hue="avg_rating",
    palette="RdYlGn", 
    hue_norm=(vmin_val, vmax_val), 
    dodge=False,
    edgecolor="black"
)

# Remove the redundant legend
if plt.gca().get_legend():
    plt.gca().get_legend().remove()

# Add a high-contrast Colorbar
norm = plt.Normalize(vmin_val, vmax_val)
sm = plt.cm.ScalarMappable(cmap="RdYlGn", norm=norm)
cbar = plt.gcf().colorbar(sm, ax=plt.gca(), pad=0.02)
cbar.set_label("Rating Sensitivity Scale (3.8 - 4.5)", weight='bold')

# Annotate Portfolio Size (Unique Titles)
# 动态计算偏移量 (最大值的 1.5%)，防止因为数据量级变大导致文字压住柱子
max_reviews = q6_plot_data['total_reviews'].max()
offset = max_reviews * 0.015

for i, row in q6_plot_data.iterrows():
    plt.text(
        row['total_reviews'] + offset, 
        i, 
        f"Titles: {int(row['total_books_published'])}", 
        va="center", 
        fontsize=11, 
        weight='semibold',
        color="#334155"
    )

plt.title("Publisher Power Matrix: High-Contrast Satisfaction Analysis", pad=25, weight='bold', fontsize=18)
plt.xlabel("Total Reader Engagement (Review Volume)", fontsize=14, weight='bold')
plt.ylabel("Publisher Name", fontsize=14, weight='bold')

# Set X-axis limit to give room for text labels
plt.xlim(0, max_reviews * 1.2)

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


q7_plot_data = pd.read_csv("Result/Q7_Temporal_Trends.csv", encoding='utf-8-sig')

q7_plot_data.columns = q7_plot_data.columns.str.strip().str.lower()

if 'unnamed: 0' in q7_plot_data.columns:
    q7_plot_data = q7_plot_data.drop(columns=['unnamed: 0'])


rename_mapping = {}
if 'year' in q7_plot_data.columns and 'review_year' not in q7_plot_data.columns:
    rename_mapping['year'] = 'review_year'
if 'review_count' in q7_plot_data.columns and 'total_reviews' not in q7_plot_data.columns:
    rename_mapping['review_count'] = 'total_reviews'

if rename_mapping:
    q7_plot_data.rename(columns=rename_mapping, inplace=True)

# 确保年份按时间顺序排列
q7_plot_data = q7_plot_data.sort_values(by='review_year').reset_index(drop=True)

# Initialize the figure and the first axis (ax1) for Review Volume
fig, ax1 = plt.subplots(figsize=(14, 8))
sns.set_context("talk", font_scale=0.9)

# 1. Plot Review Volume as Bars
sns.barplot(
    data=q7_plot_data, 
    x="review_year", 
    y="total_reviews", 
    color="#146EB4", 
    alpha=0.6, 
    ax=ax1,
    label="Total Review Volume"
)

# 2. Create a second axis (ax2) for Average Rating
ax2 = ax1.twinx()

# Plot Average Rating as a Line with markers
# Note: x=range(len(...)) is used to perfectly align the line plot with the centers of the bar plot
sns.lineplot(
    data=q7_plot_data, 
    x=range(len(q7_plot_data)), 
    y="avg_rating", 
    color="#E47911", 
    marker="o", 
    linewidth=3, 
    markersize=8,
    ax=ax2,
    label="Average Customer Rating"
)

# 3. Professional Formatting
plt.title("Temporal Evolution: Review Volume vs. Sentiment (1996-2014)", pad=20, weight='bold', fontsize=16)

# Configure Axis 1 (Volume)
ax1.set_xlabel("Review Year", fontsize=14, weight='bold')
ax1.set_ylabel("Total Review Volume", color="#146EB4", weight='bold')
ax1.tick_params(axis='y', labelcolor="#146EB4")
ax1.tick_params(axis='x', rotation=45) # 稍微倾斜年份防止重叠

# Configure Axis 2 (Rating)
ax2.set_ylabel("Average Customer Rating", color="#E47911", weight='bold')
ax2.tick_params(axis='y', labelcolor="#E47911")

# Adjust Y-axis for rating dynamically based on actual data bounds
min_rating = q7_plot_data['avg_rating'].min()
max_rating = q7_plot_data['avg_rating'].max()
ax2.set_ylim(min(4.0, min_rating - 0.1), max(4.6, max_rating + 0.1)) 

# Combine legends from both axes and move outside to avoid overlapping data
lines, labels = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax2.legend(
    lines + lines2, 
    labels + labels2, 
    loc='upper left', 
    bbox_to_anchor=(1.08, 1.0),
    frameon=True,
    edgecolor="black"
)

# Final layout adjustments
plt.subplots_adjust(right=0.82) # 给外置图例留出空间
ax1.grid(False)
ax2.grid(True, axis='y', linestyle='--', alpha=0.3)
sns.despine(right=False)

plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. 终极读取：使用 utf-8-sig 解码并直接读取 CSV
# 如果文件在 Result 文件夹下，请将路径改为 "Result/Q8_Review_Quality_Length.csv"
q8_plot_data = pd.read_csv("Result/Q8_Review_Quality_Length.csv", encoding='utf-8-sig')

# 清洗列名：去除可能存在的首尾空格，防止 KeyError
q8_plot_data.columns = q8_plot_data.columns.str.strip()

# 如果 CSV 导出了多余的索引列，安全地清理掉它
if 'Unnamed: 0' in q8_plot_data.columns:
    q8_plot_data = q8_plot_data.drop(columns=['Unnamed: 0'])

# 2. 初始化高分辨率专业画布
fig, ax1 = plt.subplots(figsize=(16, 9))
sns.set_context("talk", font_scale=0.9)

# 3. Primary Axis (ax1) for Review Volume
sns.barplot(
    data=q8_plot_data, 
    x="length_tier", 
    y="total_reviews_in_tier", 
    color="#E2E8F0", 
    alpha=0.8,
    edgecolor="#94A3B8",
    width=0.4, 
    ax=ax1,
    label="Volume of Reviews",
    legend=False
)

# 4. Secondary Axis (ax2) for Helpfulness Ratio
ax2 = ax1.twinx()

# Draw the line chart with solid markers
sns.lineplot(
    data=q8_plot_data, 
    x="length_tier", 
    y="avg_helpfulness_ratio", 
    color="#1E40AF", 
    marker="o", 
    markersize=12,
    linewidth=4,
    ax=ax2,
    label="Avg Helpfulness Ratio",
    legend=False
)

# 5. Axis Scale Isolation
ax1.set_ylim(0, q8_plot_data["total_reviews_in_tier"].max() * 1.6)
ax2.set_ylim(0.4, 0.95)

# 6. CRITICAL FIX: The 45-Degree Anchor for X-Axis Labels
# This guarantees that no matter how long the text is, they will never touch.
plt.setp(
    ax1.get_xticklabels(), 
    rotation=45, 
    ha="right", 
    rotation_mode="anchor"
)

# 7. Professional Labeling & Formatting
plt.title("Review Quality Analysis: Impact of Length on Reader Trust", pad=30, weight='bold', fontsize=20)
ax1.set_xlabel("Review Word Count Tier", fontsize=14, weight='bold', labelpad=15)
ax1.set_ylabel("Total Review Volume", color="#64748B", weight='bold')
ax2.set_ylabel("Average Helpfulness Ratio (%)", color="#1E40AF", weight='bold')

# Format the right Y-axis as percentages
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))

# 8. Annotations
for i, val in enumerate(q8_plot_data["avg_helpfulness_ratio"]):
    ax2.text(
        i, 
        val + 0.025, 
        f"{val:.1%}", 
        ha="center", 
        color="#1E40AF", 
        weight='bold',
        fontsize=13
    )

# 9. Legend Refinement
lines, labels = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax2.legend(
    lines + lines2, 
    labels + labels2, 
    loc='upper left', 
    bbox_to_anchor=(1.12, 1.0), 
    frameon=True,
    edgecolor="black"
)

# 10. Layout Optimization
plt.subplots_adjust(right=0.82, bottom=0.2)

ax1.grid(False)
ax2.grid(True, axis='y', linestyle='--', alpha=0.3)
sns.despine(right=False)

plt.show()

In [ ]:
import calendar
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


q9_plot_data = pd.read_csv("Result/Q9_Seasonality.csv", encoding='utf-8-sig')


q9_plot_data.columns = q9_plot_data.columns.str.strip().str.lower()


if 'unnamed: 0' in q9_plot_data.columns:
    q9_plot_data = q9_plot_data.drop(columns=['unnamed: 0'])


if 'month' not in q9_plot_data.columns:

    q9_plot_data.columns = ['month', 'total_reviews', 'avg_rating']


q9_plot_data['month_name'] = q9_plot_data['month'].apply(lambda x: calendar.month_abbr[int(x)])

# Initialize the figure
fig, ax1 = plt.subplots(figsize=(14, 7))
sns.set_context("talk", font_scale=0.9)

# 2. Primary Axis (ax1) for Review Volume: Area Chart
ax1.fill_between(
    q9_plot_data['month_name'], 
    q9_plot_data['total_reviews'], 
    color="#93C5FD", 
    alpha=0.5,
    label="Volume Area"
)

sns.lineplot(
    data=q9_plot_data, 
    x="month_name", 
    y="total_reviews", 
    color="#2563EB", 
    linewidth=3, 
    marker="o", 
    markersize=8,
    ax=ax1,
    label="Total Review Volume",
    legend=False
)

# 3. Secondary Axis (ax2) for Average Rating
ax2 = ax1.twinx()

sns.lineplot(
    data=q9_plot_data, 
    x="month_name", 
    y="avg_rating", 
    color="#EA580C", 
    linewidth=3, 
    marker="s", 
    markersize=8,
    ax=ax2,
    label="Average Rating",
    legend=False
)

# 4. Dynamic Peak Highlighting (Intelligent Annotation)
max_vol_idx = q9_plot_data['total_reviews'].idxmax()
max_vol_month = q9_plot_data['month_name'][max_vol_idx]
max_vol_val = q9_plot_data['total_reviews'][max_vol_idx]

ax1.annotate(
    'Peak Engagement\nSeason', 
    xy=(max_vol_idx, max_vol_val), 
    xytext=(max_vol_idx, max_vol_val * 1.15),
    ha='center',
    fontsize=12,
    weight='bold',
    color="#1E3A8A",
    arrowprops=dict(facecolor='#1E3A8A', shrink=0.05, width=2, headwidth=8)
)

# 5. Axis Limits and Scaling
ax1.set_ylim(0, q9_plot_data['total_reviews'].max() * 1.25)

# Dynamically set right y-axis based on the CSV data limits
min_rating = q9_plot_data['avg_rating'].min()
max_rating = q9_plot_data['avg_rating'].max()
ax2.set_ylim(min(4.0, min_rating - 0.05), max(4.6, max_rating + 0.05))

# 6. Professional Labeling & Formatting
plt.title("Annual Seasonality: Consumer Engagement and Satisfaction by Month", pad=25, weight='bold', fontsize=18)
ax1.set_xlabel("Month of the Year", fontsize=14, weight='bold')
ax1.set_ylabel("Total Reader Engagement (Volume)", color="#2563EB", weight='bold')
ax2.set_ylabel("Average Customer Rating", color="#EA580C", weight='bold')

# 7. Unified Legend Management
lines, labels = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()

# Filter out the area fill from the legend for a cleaner look
clean_lines = [lines[1]] + lines2 
clean_labels = [labels[1]] + labels2

ax2.legend(
    clean_lines, 
    clean_labels, 
    loc='upper left', 
    bbox_to_anchor=(1.05, 1.0), 
    frameon=True,
    edgecolor="black"
)

# 8. Layout Optimization
plt.subplots_adjust(right=0.85) 
ax1.grid(True, axis='y', linestyle='--', alpha=0.3)
ax2.grid(False)
sns.despine(right=False)

plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Load Data: Read the pre-calculated CSV directly
# Make sure "Q10_The_Hype_Factor.csv" is in your current working directory
q10_plot_data = pd.read_csv("Result/Q10_The_Hype_Factor.csv")

# Limit to the top 15 most "hyped" books for a clean visual presentation
q10_plot_data = q10_plot_data.head(15).copy()

# Clean up titles for the chart (truncate strings longer than 45 characters)
q10_plot_data['Title_Short'] = q10_plot_data['Title'].apply(
    lambda x: str(x)[:45] + '...' if str(x) != 'nan' and len(str(x)) > 45 else str(x)
)

# 2. Initialize the figure
fig, ax = plt.subplots(figsize=(14, 10))
sns.set_context("talk", font_scale=0.9)

# 3. Define a "Danger" Color Palette
# We use 'Reds_r' (Reversed Reds) so that the absolute lowest ratings appear as Dark Red
vmin_val = q10_plot_data['avg_rating'].min()
vmax_val = q10_plot_data['avg_rating'].max() 

# Ensure vmax isn't lower than 4.0 just in case to maintain the red scale warning effect
if vmax_val < 4.0:
    vmax_val = 4.0

sns.barplot(
    data=q10_plot_data, 
    x="total_reviews", 
    y="Title_Short", 
    hue="avg_rating",
    palette="Reds_r", 
    hue_norm=(vmin_val, vmax_val), 
    dodge=False,
    edgecolor="#7F1D1D", # Dark red border for emphasis
    ax=ax,
    legend=False # Disable default hue legend
)

# 4. Custom Danger Colorbar
norm = plt.Normalize(vmin_val, vmax_val)
sm = plt.cm.ScalarMappable(cmap="Reds_r", norm=norm)
cbar = plt.gcf().colorbar(sm, ax=ax, pad=0.02)
cbar.set_label("Quality Warning Scale (Darker Red = Lower Rating)", weight='bold', color="#7F1D1D")

# 5. Dual Annotations: Show both Volume and the poor Rating
for i, row in q10_plot_data.iterrows():
    # Place the rating right next to the end of the bar in bold red
    ax.text(
        row['total_reviews'] + max(q10_plot_data['total_reviews']) * 0.01, # Dynamic offset
        i, 
        f"  Rating: {row['avg_rating']:.2f}", 
        va="center", 
        fontsize=12, 
        weight='bold',
        color="#B91C1C"
    )

# 6. Professional Labeling & Formatting
plt.title("The Hype Factor: High Market Dominance vs. Low Reader Satisfaction", pad=25, weight='bold', fontsize=18, color="#7F1D1D")
ax.set_xlabel("Total Reader Engagement (Review Volume)", fontsize=14, weight='bold')
ax.set_ylabel("Book Titles (Truncated)", weight='bold')

# Expand X-axis slightly to make room for the text annotations
ax.set_xlim(0, q10_plot_data['total_reviews'].max() * 1.15)

# 7. Layout Optimization
plt.subplots_adjust(left=0.3) # Give plenty of room for long book titles on the left
ax.grid(True, axis='x', linestyle='--', alpha=0.3)
sns.despine()

plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# 1. Load and Merge Data (Assuming files are in Result/ folder)
try:
    # Use utf-8-sig to handle potential BOM issues
    q1_data = pd.read_csv("Result/Q1_Category_Preference.csv", encoding='utf-8-sig')
    q4_data = pd.read_csv("Result/Q4_Pricing_Strategy.csv", encoding='utf-8-sig')
    
    # Standardize column names
    q1_data.columns = q1_data.columns.str.strip().str.lower()
    q4_data.columns = q4_data.columns.str.strip().str.lower()
    
    # Rename to align keys
    q1_data = q1_data.rename(columns={'single_category': 'category', 'count': 'volume', 'total_reviews': 'volume'})
    q4_data = q4_data.rename(columns={'genre': 'category'})
    
    # Merge datasets
    merged_data = pd.merge(q1_data, q4_data, on='category')
except Exception as e:
    print(f"Error: {e}. Please ensure CSV files are in the 'Result/' directory.")

# 2. Setup Plot with Professional Aesthetics
plt.figure(figsize=(16, 12))
sns.set_style("white") # Clean white background
sns.set_context("talk")

# 3. BACKGROUND LAYER (zorder=1): Strategic Quadrants as Watermarks
# Using median to split the chart into 4 strategic zones
price_mid = merged_data['avg_price'].median()
rating_mid = merged_data['avg_rating'].median()

# Draw the cross-hair lines
plt.axvline(price_mid, color='#E2E8F0', linestyle='--', linewidth=2, zorder=1)
plt.axhline(rating_mid, color='#E2E8F0', linestyle='--', linewidth=2, zorder=1)

# Quadrant labels as light-colored "Watermarks" to avoid clutter
watermark_style = {'fontsize': 28, 'color': '#F1F5F9', 'weight': 'black', 'alpha': 0.8, 'zorder': 1}
plt.text(merged_data['avg_price'].max(), merged_data['avg_rating'].max(), 'PREMIUM STARS', ha='right', va='top', **watermark_style)
plt.text(merged_data['avg_price'].min(), merged_data['avg_rating'].max(), 'VALUE LEADERS', ha='left', va='top', **watermark_style)
plt.text(merged_data['avg_price'].max(), merged_data['avg_rating'].min(), 'OVERPRICED', ha='right', va='bottom', **watermark_style)
plt.text(merged_data['avg_price'].min(), merged_data['avg_rating'].min(), 'LOW-END', ha='left', va='bottom', **watermark_style)

# 4. MIDDLE LAYER (zorder=2): The Bubbles
# Color mapped to rating, Size mapped to volume
scatter = plt.scatter(
    x=merged_data['avg_price'],
    y=merged_data['avg_rating'],
    s=merged_data['volume'] / merged_data['volume'].max() * 4500, # Normalize bubble size
    c=merged_data['avg_rating'],
    cmap="RdYlGn",
    alpha=0.6,
    edgecolors="white", # White border makes bubbles "pop"
    linewidth=2,
    zorder=2
)

# 5. TOP LAYER (zorder=3): Smart Labels
# Label the Top 15 categories by volume to provide high coverage without total mess
top_categories = merged_data.nlargest(15, 'volume')

for i, row in top_categories.iterrows():
    plt.annotate(
        row['category'],
        xy=(row['avg_price'], row['avg_rating']),
        xytext=(8, 8), # Offset text away from the center of the bubble
        textcoords='offset points',
        fontsize=11,
        weight='bold',
        color='#334155',
        # Semi-transparent box background for readability
        bbox=dict(facecolor='white', alpha=0.7, edgecolor='#F1F5F9', boxstyle='round,pad=0.2'),
        zorder=3
    )

# 6. Final Formatting
plt.title("Amazon Books: Category Strategic Positioning Matrix", fontsize=24, weight='bold', pad=30, color='#1E293B')
plt.xlabel("Average Book Price (USD) → Profitability", fontsize=16, labelpad=15, weight='bold')
plt.ylabel("Average Customer Rating → Satisfaction", fontsize=16, labelpad=15, weight='bold')

# Colorbar to explain the rating heat
cbar = plt.colorbar(scatter)
cbar.set_label("Rating Score (1-5)", rotation=270, labelpad=20, weight='bold')

# Adjust limits to give some breathing room
plt.xlim(merged_data['avg_price'].min() * 0.85, merged_data['avg_price'].max() * 1.1)
plt.ylim(merged_data['avg_rating'].min() - 0.05, merged_data['avg_rating'].max() + 0.05)

sns.despine() # Remove top and right borders
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql import functions as F

# 1. Prepare the data with correct column names
# We extract the first element from 'categories_array' as 'category'
df_heatmap = df_final.withColumn("category", F.col("categories_array").getItem(0)) \
                     .filter(F.col("category").isNotNull() & F.col("publisher").isNotNull())

# 2. Identify the Top 10 Publishers and Top 10 Categories for a clean heatmap
top_publishers_list = df_heatmap.groupBy("publisher").count().orderBy(F.desc("count")).limit(10).select("publisher").rdd.flatMap(lambda x: x).collect()
top_categories_list = df_heatmap.groupBy("category").count().orderBy(F.desc("count")).limit(10).select("category").rdd.flatMap(lambda x: x).collect()

# 3. Filter and Aggregate for the heatmap
heatmap_data_spark = df_heatmap.filter(
    (F.col("publisher").isin(top_publishers_list)) & 
    (F.col("category").isin(top_categories_list))
).groupBy("publisher", "category").agg(F.count("*").alias("review_count"))

# 4. Convert to Pandas and Pivot
heatmap_df = heatmap_data_spark.toPandas()
pivot_table = heatmap_df.pivot(index='publisher', columns='category', values='review_count').fillna(0)

# 5. Plotting the Matrix
plt.figure(figsize=(16, 10))
sns.set_context("talk")

# Use 'YlGnBu' for a high-end corporate feel
sns.heatmap(
    pivot_table, 
    annot=True, 
    fmt=".0f", 
    cmap="YlGnBu", 
    linewidths=.5, 
    cbar_kws={'label': 'Market Influence (Review Volume)'},
    annot_kws={"size": 11, "weight": "bold"}
)

# 6. Professional Labeling (English Only)
plt.title("Market Dominance Matrix: Publisher Niche Analysis", pad=30, weight='bold', fontsize=22)
plt.xlabel("Book Category / Genre", fontsize=14, weight='bold', labelpad=15)
plt.ylabel("Top 10 Publishers", fontsize=14, weight='bold', labelpad=15)

# Rotate labels for perfect readability
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pyspark.sql import functions as F

# 1. Spark Multi-Dimensional Aggregation
# Extract category and calculate 5 key metrics per genre
radar_spark_df = df_final.withColumn("category", F.col("categories_array").getItem(0)) \
    .filter(F.col("category").isNotNull()) \
    .groupBy("category") \
    .agg(
        F.count("*").alias("volume"),
        F.avg("rating").alias("satisfaction"),
        F.avg("Price").alias("premiumness"),
        # Calculate Reliability as (1 / standard deviation of rating)
        # We use a small epsilon to avoid division by zero
        (F.lit(1) / (F.stddev("rating") + 0.01)).alias("reliability"),
        # Calculate Trust as Average Helpfulness Ratio
        F.avg(F.col("helpful_votes") / (F.col("total_votes") + 1)).alias("trust")
    )

# 2. Focus on the Top 5 Categories by Volume for the best visual clarity
top_5_categories = radar_spark_df.orderBy(F.desc("volume")).limit(5).toPandas()

# 3. Normalization (Scale all metrics to 0.0 - 1.0 for the Radar chart)
# This is crucial so that Price and Rating can be compared on the same scale
metrics = ["volume", "satisfaction", "premiumness", "reliability", "trust"]
radar_norm = top_5_categories.copy()

for metric in metrics:
    max_val = radar_norm[metric].max()
    min_val = radar_norm[metric].min()
    radar_norm[metric] = (radar_norm[metric] - min_val) / (max_val - min_val)

# 4. Radar Chart Construction
labels = ["Popularity", "Satisfaction", "Premiumness", "Reliability", "Trust"]
num_vars = len(labels)

# Compute angle for each axis
angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
angles += angles[:1] # Close the loop

fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))

# Color palette for different categories
colors = plt.cm.get_cmap("Set1", len(radar_norm))

for i, row in radar_norm.iterrows():
    values = row[metrics].tolist()
    values += values[:1] # Close the loop
    
    # Draw the line and fill the area
    ax.plot(angles, values, color=colors(i), linewidth=3, label=row['category'], alpha=0.8)
    ax.fill(angles, values, color=colors(i), alpha=0.15)

# 5. Professional Aesthetics & Formatting
ax.set_theta_offset(np.pi / 2)
ax.set_theta_direction(-1)

# Set labels for each spoke
ax.set_thetagrids(np.degrees(angles[:-1]), labels, weight='bold', fontsize=12)

# Professional styling: Grid lines and background
ax.grid(color='#CBD5E1', linestyle='--', linewidth=0.8)
plt.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), frameon=True, edgecolor="black")

plt.title("Market Portfolio DNA: Strategic Multi-Dimensional Profile", pad=40, weight='bold', fontsize=20)

plt.show()

In [ ]:
# 1. Spark Aggregation: Calculate Helpfulness Ratio per Star Rating
helpfulness_bias = df_final.groupBy("rating") \
    .agg(
        F.count("*").alias("review_count"),
        F.avg(F.col("helpful_votes") / (F.col("total_votes") + 1)).alias("avg_helpfulness_ratio")
    ) \
    .orderBy("rating").toPandas()

# 2. Plotting: The "Bias" Curve
plt.figure(figsize=(12, 7))
sns.set_context("talk")

# Use a custom color list to represent stars
star_colors = ["#EF4444", "#F59E0B", "#FBBF24", "#34D399", "#059669"]

# Create a combined Bar + Line Chart
ax = sns.barplot(data=helpfulness_bias, x="rating", y="avg_helpfulness_ratio", palette=star_colors, alpha=0.8)

# Add a trend line to show the bias
sns.lineplot(data=helpfulness_bias, x=range(5), y="avg_helpfulness_ratio", marker="o", color="black", linewidth=3, markersize=10)

# 3. Formatting
plt.title("The Helpfulness Bias: Which Star Ratings Do Readers Trust Most?", pad=25, weight='bold', fontsize=18)
plt.xlabel("Star Rating (1 = Hated, 5 = Loved)", weight='bold')
plt.ylabel("Average Helpfulness Ratio (%)", weight='bold')

# Add percentage labels
for p in ax.patches:
    ax.annotate(f'{p.get_height():.1%}', (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='center', xytext=(0, 10), textcoords='offset points', weight='bold')

plt.ylim(0, helpfulness_bias['avg_helpfulness_ratio'].max() * 1.2)
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql import functions as F
import numpy as np

# 1. Spark Aggregation (Standard Logic)
consistency_df = df_final.withColumn("category", F.col("categories_array").getItem(0)) \
    .filter(F.col("category").isNotNull()) \
    .groupBy("category") \
    .agg(
        F.count("*").alias("volume"),
        F.avg("rating").alias("avg_rating"),
        F.stddev("rating").alias("rating_stddev")
    ) \
    .filter(F.col("volume") > 300)

plot_data = consistency_df.toPandas()

# 2. Calculate the "Center of Gravity" of the chart
x_mid = plot_data['avg_rating'].median()
y_mid = plot_data['rating_stddev'].median()

# 3. Initialize Figure
plt.figure(figsize=(20, 14)) # Massive canvas for the "Exploded" look
sns.set_style("white")

# Watermark Background
plt.axvline(x_mid, color='#F1F5F9', linestyle='-', linewidth=4, zorder=1)
plt.axhline(y_mid, color='#F1F5F9', linestyle='-', linewidth=4, zorder=1)

# 4. Draw the Bubbles
scatter = plt.scatter(
    x=plot_data['avg_rating'],
    y=plot_data['rating_stddev'],
    s=plot_data['volume'] / plot_data['volume'].max() * 5000,
    c=plot_data['avg_rating'],
    cmap="RdYlGn",
    alpha=0.3, # Faded bubbles to make labels and lines the hero
    edgecolors="white",
    linewidth=2,
    zorder=2
)

# 5. 【CORE FIX】Circular Explosion Labeling Algorithm
# Take the Top 15 categories by volume
top_15 = plot_data.nlargest(15, 'volume').copy()

# Calculate the angle of each point relative to the center
top_15['angle_from_center'] = np.arctan2(
    top_15['rating_stddev'] - y_mid, 
    top_15['avg_rating'] - x_mid
)

# Sort them by angle so they map cleanly to the "Clock Face"
top_15 = top_15.sort_values('angle_from_center').reset_index()

# Define the "Halo" - 15 evenly spaced points on a circle
# radius_pts determines how far the labels are "pushed" out (higher = longer lines)
radius_pts = 180 
target_angles = np.linspace(0, 2 * np.pi, 15, endpoint=False)

for i, row in top_15.iterrows():
    # Target position on the halo
    target_angle = target_angles[i]
    
    # Calculate text offset based on the target angle
    # This forces the labels to fan out 360 degrees
    text_x = radius_pts * np.cos(target_angle)
    text_y = radius_pts * np.sin(target_angle)
    
    # Adjust horizontal alignment based on side of the chart
    align = "left" if text_x > 0 else "right"
    
    plt.annotate(
        row['category'],
        xy=(row['avg_rating'], row['rating_stddev']),
        xytext=(text_x, text_y),
        textcoords='offset points',
        fontsize=12,
        weight='bold',
        color='#0F172A',
        ha=align,
        va='center',
        bbox=dict(facecolor='white', alpha=0.9, edgecolor='#CBD5E1', boxstyle='round,pad=0.5'),
        arrowprops=dict(
            arrowstyle="->",
            # connectionstyle "bar" or "arc3" with high rad creates the "long reach" effect
            connectionstyle=f"arc3,rad={0.2 if i%2==0 else -0.2}", 
            color='#94A3B8',
            lw=1.5,
            alpha=0.5
        ),
        zorder=4
    )

# 6. Professional Polish
plt.title("Strategic Consistency Matrix: Exploded Portfolio View", pad=50, weight='black', fontsize=28, color='#0F172A')
plt.xlabel("Average Market Rating (Quality) →", weight='bold', fontsize=18, labelpad=20)
plt.ylabel("Rating Standard Deviation (Consumer Disagreement) →", weight='bold', fontsize=18, labelpad=20)

# Expand limits significantly to accommodate the long "Explosion" lines
plt.xlim(plot_data['avg_rating'].min() - 0.4, plot_data['avg_rating'].max() + 0.4)
plt.ylim(plot_data['rating_stddev'].min() - 0.2, plot_data['rating_stddev'].max() + 0.2)

# Legend-style colorbar
cbar = plt.colorbar(scatter, fraction=0.02, pad=0.08)
cbar.set_label('Rating Intensity Scale', weight='bold', fontsize=14)

sns.despine(left=True, bottom=True)
plt.tight_layout()
plt.show()

In [ ]:
# 1. 过滤掉价格过高的离群值
hex_data = df_final.filter(F.col("Price") < 100).select(
    "Price", 
    (F.col("helpful_votes") / (F.col("total_votes") + 1)).alias("helpfulness_ratio")
).toPandas()

# 2. 绘制 Hexbin 图
plt.figure(figsize=(14, 10))

# gridsize 控制六边形的大小，越大越精细
hb = plt.hexbin(
    hex_data['Price'], 
    hex_data['helpfulness_ratio'], 
    gridsize=30, 
    cmap='YlGnBu', 
    mincnt=1, # 只显示有数据的格
    edgecolors='white',
    linewidths=0.1
)

# 3. 添加颜色条
cb = plt.colorbar(hb)
cb.set_label('Density of Reviews (Concentration)', weight='bold')

# 4. 加上趋势线（Lowess 曲线，展示价格与有用性的关系）
sns.regplot(
    data=hex_data.sample(2000), # 采样 2000 个点画趋势，防止渲染过慢
    x="Price", 
    y="helpfulness_ratio", 
    scatter=False, 
    color="#EF4444", 
    lowess=True
)

plt.title("The Knowledge Intensity Map: Price vs. Review Helpfulness", pad=25, weight='bold', fontsize=18)
plt.xlabel("Book Price (USD)", weight='bold')
plt.ylabel("Reader Helpfulness Ratio (%)", weight='bold')

sns.despine()
plt.show()